# Flash Texter 단계 2 — Qwen3.5-0.8B LoRA 파인튜닝 (Colab)

사전학습된 Qwen3.5-0.8B-Base를 우리가 직접 작성한 대화 데이터로 LoRA 파인튜닝합니다.

**중요**: 이 노트북은 huggingface.co 접근이 필요합니다 (로컬 개발 샌드박스에서는 네트워크 제약으로 실행 검증이 불가능했습니다 — `docs/15-stage2-pretrained-finetuning.md` 5절 참조). Colab은 인터넷 제한이 없으므로 여기서 실제로 실행해 검증합니다.

**순서**: 1) 리포지토리 클론 2) GPU 확인 3) 라이브러리 설치 4) Drive 마운트 5) 데이터 생성 6) LoRA 파인튜닝 7) 추론 테스트 8) HF Hub 업로드

## 1. 리포지토리 클론

In [ ]:
!git clone https://github.com/choichoi3227-crypto/cloud-press.git
%cd cloud-press/ai-models/training/flash-texter

## 2. GPU 확인
런타임 → 런타임 유형 변경 → T4 GPU로 설정한 뒤 실행하세요.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (경고: 매우 느립니다)")

## 3. 라이브러리 설치

In [ ]:
# Qwen3.5는 transformers >= 5.2.0 부터 지원된다 (실제 조사로 확인, huggingface/transformers
# v5.2.0 릴리스 노트 기준). 그 이전 버전에서는 모델 자체가 로드되지 않는다.
!pip install -q "transformers>=5.2.0" accelerate "peft>=0.14.0" datasets bitsandbytes

## 4. Google Drive 마운트 (체크포인트 저장용)
Colab 세션이 끊겨도 학습을 이어갈 수 있도록, 체크포인트는 반드시 Drive에 저장합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = "/content/drive/MyDrive/cloud-press/checkpoints/flash-texter-lora"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("체크포인트 저장 위치:", CHECKPOINT_DIR)

## 5. 파인튜닝 데이터 생성
직접 작성한 템플릿 조합으로 데이터를 생성합니다. 현재 규모(약 900개 고유 조합)는 LoRA 파인튜닝의 최소 실험 규모입니다. `training/flash-texter/generate_finetune_data.py`의 `CATEGORIES`에 표현을 더 추가하면 규모를 늘릴 수 있습니다 (docs/15 3.1절 참조).

In [ ]:
DATA_PATH = "/content/drive/MyDrive/cloud-press/data/flash_texter_finetune.jsonl"
import os
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
!python generate_finetune_data.py --out {DATA_PATH} --repeat-per-template 300

## 6. LoRA 파인튜닝 실행
체크포인트가 있으면 자동으로 이어서 학습합니다. 베이스 모델은 최초 실행 시 자동 다운로드됩니다.

In [ ]:
!python train_lora.py \
    --data {DATA_PATH} \
    --base-model Qwen/Qwen3.5-0.8B-Base \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --epochs 3 \
    --batch-size 4

## 7. 직접 대화해보기 (정성 평가)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen3.5-0.8B-Base"
ADAPTER_DIR = f"{CHECKPOINT_DIR}/final"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

def chat(prompt, max_new_tokens=200):
    text = f"<|im_start|>system\n당신은 Cloud Press가 직접 학습시킨 한국어 대화 모델 Flash Texter입니다. 친절하고 정확하게 답하세요.<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
    return tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

for q in ["안녕하세요", "머신러닝이 뭐야?", "시간 관리 어떻게 해?", "오늘 날씨 어때?"]:
    print(f"Q: {q}")
    print(f"A: {chat(q)}")
    print()

## 8. CPU 추론 속도 실측 (중요 — docs/15 2.2절의 미확정 항목을 채우는 실측)

In [ ]:
import time

# 알려진 버그 우회: AutoModelForCausalLM.from_pretrained(..., dtype=...)로 Qwen3.5
# 계열을 로드하면 dtype 인자가 조용히 무시되고 항상 bfloat16으로 로드되는 문제가
# 보고되어 있다 (huggingface/transformers issue #46459). CPU 추론 속도를 정확히 재려면
# 실제로 float32로 로드됐는지 확인이 필요하므로, 로드 직후 dtype을 검증한다.
cpu_model = base_model.to("cpu").float()

actual_dtype = next(cpu_model.parameters()).dtype
print(f"모델 파라미터 실제 dtype: {actual_dtype}")
if actual_dtype != torch.float32:
    print(
        "경고: float32로 변환을 시도했지만 실제로는 다른 dtype입니다. "
        "알려진 dtype 무시 버그(transformers #46459)에 해당할 수 있습니다. "
        "측정된 속도가 실제 float32 추론 속도가 아닐 수 있으니 유의하세요."
    )

cpu_peft_model = PeftModel.from_pretrained(cpu_model, ADAPTER_DIR)

test_prompt = "머신러닝이 뭐야?"
text = f"<|im_start|>user\n{test_prompt}<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer(text, return_tensors="pt")

start = time.time()
with torch.no_grad():
    output = cpu_peft_model.generate(**inputs, max_new_tokens=100)
elapsed = time.time() - start

print(f"CPU 추론 시간 (100 토큰): {elapsed:.2f}초")
print(f"토큰당 평균: {elapsed/100*1000:.1f}ms")
print("이 수치를 docs/15-stage2-pretrained-finetuning.md 2.1절에 실측치로 반영하세요.")
print(tokenizer.decode(output[0], skip_special_tokens=True))

## 9. Hugging Face Hub에 LoRA 어댑터 업로드
LoRA 어댑터만 업로드합니다 (베이스 모델 전체가 아니라 수십 MB의 어댑터 가중치만).

In [ ]:
from huggingface_hub import login, HfApi

login()

HF_REPO_ID = "<your-username>/flash-texter-lora"  # 실제 사용자명으로 변경

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, exist_ok=True)
api.upload_folder(
    folder_path=f"{CHECKPOINT_DIR}/final",
    repo_id=HF_REPO_ID,
)
print("업로드 완료:", HF_REPO_ID)